# 正解 — Must 02 — 階層回帰（くちばし＋ひれ）

体重`Body_Mass`をくちばし長`Culmen_Length`とひれ長`Flipper_Length`で説明し、切片`intercept`・傾き`beta_culmen` / `beta_flipper`に種差を入れた階層モデルです（例01にculmenを追加した拡張）。

`python/` に入って実行してください（データは `../../../data/`）。


## 準備


In [ ]:
import arviz as az
import arviz_plots as azp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
from arviz_plots import plot_trace_dist
from IPython.display import display

azp.style.use("arviz-variat")
SEED = 123

df = pd.read_parquet("../../../data/penguins.parquet").dropna(
    subset=["Body_Mass", "Culmen_Length", "Flipper_Length", "Species_short"]
)


## データとcoords

`Species_short`を`factorize`し、culmen / flipper / species_idx と `coords` を用意してください。


In [ ]:
df["species_idx"], species_levels = pd.factorize(df["Species_short"].astype(str))
culmen = df["Culmen_Length"].to_numpy()
flipper = df["Flipper_Length"].to_numpy()
species_idx = df["species_idx"].to_numpy()

coords = {
    "species": list(species_levels),
    "obs": np.arange(len(df)),
}


## モデル・graphviz・サンプリング

PyMCで階層モデルを定義し、`model_to_graphviz`をノートブックに表示してからサンプリングしてください（`random_seed=123`）。


In [ ]:
with pm.Model(coords=coords) as model:
    # くちばし・ひれは mm のままなので、切片は 0 mm への外挿になりうる（弱情報事前）
    intercept_all = pm.Normal("intercept_all", 0, 5000)
    beta_culmen_all = pm.Normal("beta_culmen_all", 0, 50)
    beta_flipper_all = pm.Normal("beta_flipper_all", 0, 20)

    sigma_intercept = pm.HalfNormal("sigma_intercept", 1000)
    sigma_culmen = pm.HalfNormal("sigma_culmen", 30)
    sigma_flipper = pm.HalfNormal("sigma_flipper", 10)

    intercept = pm.Normal("intercept", intercept_all, sigma_intercept, dims="species")
    beta_culmen = pm.Normal("beta_culmen", beta_culmen_all, sigma_culmen, dims="species")
    beta_flipper = pm.Normal(
        "beta_flipper", beta_flipper_all, sigma_flipper, dims="species"
    )

    mu_n = (
        intercept[species_idx]
        + beta_culmen[species_idx] * culmen
        + beta_flipper[species_idx] * flipper
    )
    sigma_y = pm.HalfNormal("sigma_y", 400)
    pm.Normal("y", mu=mu_n, sigma=sigma_y, observed=df["Body_Mass"].to_numpy(), dims="obs")

    graph = pm.model_to_graphviz(model)
    display(graph)

    idata = pm.sample(
        draws=500,
        tune=500,
        chains=4,
        target_accept=0.99,
        random_seed=SEED,
        progressbar=True,
    )


## 要約と診断

ハイパーパラメータの`az.summary`とR-hatを表示してください。


In [ ]:
summary = az.summary(
    idata,
    var_names=[
        "intercept_all",
        "beta_culmen_all",
        "beta_flipper_all",
        "sigma_intercept",
        "sigma_culmen",
        "sigma_flipper",
        "sigma_y",
    ],
    round_to=2,
)
print(f"{summary=}")
max_rhat = float(summary["r_hat"].max())
if max_rhat > 1.05:
    print(
        f"NOTE: {max_rhat=:.3f} (>1.05). "
        "デモ設定のため不安定なことがあります。講師デモを参照して構いません。"
    )
else:
    print(f"R-hat OK ({max_rhat=:.3f})")
print(f"{list(species_levels)=}, {len(df)=}")


## 種ごとの係数

`intercept` / `beta_culmen` / `beta_flipper`の事後中央値と94% ETI、および種ごとの回帰式（中央値）を表示してください。


In [ ]:
coef_summary = az.summary(
    idata,
    var_names=["intercept", "beta_culmen", "beta_flipper"],
    kind="all_median",
    ci_prob=0.94,
    round_to=1,
)
print("\ncoefficients (posterior median + 94% ETI):")
print(f"{coef_summary=}")

print("\nspecies regression (posterior median):")
print("Body_Mass ≈ intercept + beta_culmen·Culmen + beta_flipper·Flipper")
for sp in species_levels:
    intercept_m = float(idata.posterior["intercept"].sel(species=sp).median())
    culmen_m = float(idata.posterior["beta_culmen"].sel(species=sp).median())
    flipper_m = float(idata.posterior["beta_flipper"].sel(species=sp).median())
    print(f"  {sp}: {intercept_m:.0f} + {culmen_m:.1f}·Culmen + {flipper_m:.1f}·Flipper")


## forest

`az.plot_forest`を、intercept と slopes（beta_culmen·beta_flipper）の2枚に分けてノートブックに表示してください。


In [ ]:
pc = az.plot_forest(
    idata,
    var_names=["intercept"],
    combined=True,
    backend="matplotlib",
    figure_kwargs={"figsize": (9, 2.8)},
)
pc.add_title("Posterior: species-specific intercept")
pc.show()

pc = az.plot_forest(
    idata,
    var_names=["beta_culmen", "beta_flipper"],
    combined=True,
    backend="matplotlib",
    figure_kwargs={"figsize": (9, 4.2)},
)
pc.add_title("Posterior: species-specific slopes (culmen, flipper)")
pc.show()


## 事後分布

`az.plot_dist`を、intercept と slopes（beta_culmen·beta_flipper）の2枚に分けてノートブックに表示してください。


In [ ]:
pc = az.plot_dist(
    idata,
    var_names=["intercept"],
    ci_prob=0.94,
    backend="matplotlib",
    figure_kwargs={"figsize": (9, 2.8)},
)
pc.show()

pc = az.plot_dist(
    idata,
    var_names=["beta_culmen", "beta_flipper"],
    ci_prob=0.94,
    backend="matplotlib",
    figure_kwargs={"figsize": (9, 4.5)},
)
pc.show()


## trace_dist

`plot_trace_dist`を、intercept と slopes（beta_culmen·beta_flipper）の2枚に分けてノートブックに表示してください。


In [ ]:
pc = plot_trace_dist(
    idata, var_names=["intercept"], backend="matplotlib", compact=True
)
pc.show()

pc = plot_trace_dist(
    idata,
    var_names=["beta_culmen", "beta_flipper"],
    backend="matplotlib",
    compact=True,
)
pc.show()
